# บทที่ 7: โครงข่ายประสาทแบบวนซ้ำ (Recurrent Neural Networks: RNN, LSTM, GRU)

ใน Notebook นี้ เราจะสร้าง RNN, LSTM และ GRU จากศูนย์ พร้อมทั้งศึกษาปัญหาเกรเดียนต์สูญหายและเกรเดียนต์ระเบิด (Vanishing / Exploding Gradient)

**ศัพท์ที่สำคัญในบทนี้:**- เวกเตอร์ (vector) — อาร์เรย์หนึ่งมิติ- เมทริกซ์ (matrix) — อาร์เรย์สองมิติ- ค่าน้ำหนัก (weight) — พารามิเตอร์ที่ปรับได้- ค่าไบแอส (bias) — ค่าเลื่อน- อินพุต (input) — ข้อมูลนำเข้า- เอาต์พุต (output) — ผลลัพธ์- ค่าสูญเสีย (loss) — วัดความคลาดเคลื่อน- เกรเดียนต์ (gradient) — ทิศทางการปรับ- ฟังก์ชันกระตุ้น (activation function) — ฟังก์ชันไม่เป็นเชิงเส้น- โครงข่ายประสาทเทียม (neural network) — โมเดลแมชชีนเลิร์นนิง

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

np.random.seed(42)

## 2. การสร้าง RNN อย่างง่าย (Simple RNN)

In [ ]:
class SimpleRNN:
    """
    การสร้าง RNN อย่างง่ายจากศูนย์

    h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
    y_t = softmax(W_hy @ h_t + b_y)
    """

    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size

        # กำหนดค่าน้ำหนักเริ่มต้น
        self.W_xh = np.random.randn(hidden_size, input_size) * 0.01
        self.W_hh = np.random.randn(hidden_size, hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))

        self.W_hy = np.random.randn(output_size, hidden_size) * 0.01
        self.b_y = np.zeros((output_size, 1))

    def forward(self, x_sequence):
        """
        การส่งผ่านสัญญาณไปข้างหน้าตลอดลำดับ

        พารามิเตอร์:
        - x_sequence: list ของอินพุต [x_1, x_2, ..., x_T]
        """
        h = np.zeros((self.hidden_size, 1))
        self.hidden_states = [h]
        self.inputs = []

        outputs = []

        for x in x_sequence:
            x = x.reshape(-1, 1)
            self.inputs.append(x)

            # สถานะแฝง
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            self.hidden_states.append(h)

            # เอาต์พุต
            y = self.W_hy @ h + self.b_y
            outputs.append(y)

        return outputs

    def predict(self, x_sequence):
        """ดูเอาต์พุตสุดท้าย"""
        outputs = self.forward(x_sequence)
        return outputs[-1]

## 3. ทดสอบ Simple RNN

In [ ]:
# สร้าง RNN
rnn = SimpleRNN(input_size=2, hidden_size=4, output_size=1)

# ลำดับทดสอบ
sequence = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1])
]

outputs = rnn.forward(sequence)

print("=== การส่งผ่านสัญญาณไปข้างหน้าของ RNN ===")
for t, (x, h, y) in enumerate(zip(sequence, rnn.hidden_states[1:], outputs)):
    print(f"\nขั้นเวลาที่ {t+1}:")
    print(f"  อินพุต: {x}")
    print(f"  สถานะแฝง: {h.flatten()}")
    print(f"  เอาต์พุต: {y.flatten()}")

## 4. สาธิตเกรเดียนต์สูญหายและเกรเดียนต์ระเบิด (Vanishing / Exploding Gradient)

In [ ]:
def demonstrate_vanishing_gradient():
    """
    สาธิตปัญหาเกรเดียนต์สูญหายและเกรเดียนต์ระเบิด

    ในโครงข่ายแบบวนซ้ำ เกรเดียนต์ไหลย้อนกลับผ่านหลายขั้นเวลา
    ตัวคูณต่อขั้นที่น้อยกว่า 1 ทำให้เกรเดียนต์หดตัวจนสูญหาย
    ส่วนตัวคูณที่มากกว่า 1 ทำให้เกรเดียนต์ขยายตัวจนระเบิด
    """
    time_steps = 20

    # จำลองการไหลของเกรเดียนต์ด้วยตัวคูณต่อขั้น 0.8 ตามตัวอย่าง BPTT ในบท (< 1 → สูญหาย)
    gradients_tanh = [1.0]
    for t in range(time_steps):
        grad = gradients_tanh[-1] * 0.8
        gradients_tanh.append(grad)

    # จำลองการไหลของเกรเดียนต์ด้วย ReLU (ตัวคูณต่อขั้น = 1 เมื่อทำงาน)
    gradients_relu = [1.0]
    for t in range(time_steps):
        grad = gradients_relu[-1] * 1.0
        gradients_relu.append(grad)

    # จำลองกรณีเกรเดียนต์ระเบิด (ตัวคูณต่อขั้น 1.5 ตามตัวอย่างในบท)
    gradients_explode = [1.0]
    for t in range(time_steps):
        grad = gradients_explode[-1] * 1.5
        gradients_explode.append(grad)

    plt.figure(figsize=(10, 6))
    plt.semilogy(gradients_tanh, 'r-o', label='Tanh (เกรเดียนต์สูญหาย, ตัวคูณ 0.8)')
    plt.semilogy(gradients_relu, 'b-s', label='ReLU (คงที่, ตัวคูณ 1.0)')
    plt.semilogy(gradients_explode, 'g-^', label='เกรเดียนต์ระเบิด (ตัวคูณ 1.5)')
    plt.xlabel('ขั้นเวลา')
    plt.ylabel('ขนาดของเกรเดียนต์ (สเกล log)')
    plt.title('เกรเดียนต์สูญหายและเกรเดียนต์ระเบิดใน RNN')
    plt.legend()
    plt.show()

    print(f"\nเกรเดียนต์ของ Tanh หลัง 20 ขั้น: {gradients_tanh[-1]:.6f} (บทคำนวณ 0.8^20 ≈ 0.012)")
    print(f"เกรเดียนต์ของ ReLU หลัง 20 ขั้น: {gradients_relu[-1]:.6f}")
    print(f"เกรเดียนต์ที่ระเบิดหลัง 20 ขั้น: {gradients_explode[-1]:.2f}")
    print("หมายเหตุ: การคลิปเกรเดียนต์ (gradient clipping) ช่วยจำกัดกรณีระเบิดได้ แต่ไม่ช่วยแก้ปัญหาสูญหายโดยตรง")

demonstrate_vanishing_gradient()

## 5. การสร้างเซลล์ LSTM (LSTM Cell)

In [ ]:
class LSTMCell:
    """
    เซลล์ LSTM ที่มี 3 เกต:
    - เกตลืม (Forget Gate): f_t = σ(W_f @ [h_{t-1}, x_t] + b_f)
    - เกตอินพุต (Input Gate): i_t = σ(W_i @ [h_{t-1}, x_t] + b_i)
    - เกตเอาต์พุต (Output Gate): o_t = σ(W_o @ [h_{t-1}, x_t] + b_o)

    การอัปเดตสถานะเซลล์:
    - C̃_t = tanh(W_c @ [h_{t-1}, x_t] + b_c)
    - C_t = f_t * C_{t-1} + i_t * C̃_t
    - h_t = o_t * tanh(C_t)
    """

    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size

        # เกตลืม
        self.W_f = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_f = np.zeros((hidden_size, 1))

        # เกตอินพุต
        self.W_i = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_i = np.zeros((hidden_size, 1))

        # เกตเอาต์พุต
        self.W_o = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_o = np.zeros((hidden_size, 1))

        # สถานะเซลล์ (ตัวเลือก)
        self.W_c = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_c = np.zeros((hidden_size, 1))

    def forward(self, x, h_prev, c_prev):
        """คำนวณหนึ่งขั้นเวลา"""
        # เชื่อม h_prev กับ x เข้าด้วยกัน
        combined = np.vstack([h_prev, x.reshape(-1, 1)])

        # เกตต่างๆ
        f_t = self._sigmoid(self.W_f @ combined + self.b_f)  # เกตลืม
        i_t = self._sigmoid(self.W_i @ combined + self.b_i)  # เกตอินพุต
        o_t = self._sigmoid(self.W_o @ combined + self.b_o)  # เกตเอาต์พุต

        # สถานะเซลล์ตัวเลือก
        c_tilde = np.tanh(self.W_c @ combined + self.b_c)

        # อัปเดตสถานะเซลล์
        c_t = f_t * c_prev + i_t * c_tilde

        # อัปเดตสถานะแฝง
        h_t = o_t * np.tanh(c_t)

        return h_t, c_t, (f_t, i_t, o_t, c_tilde)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


class LSTM:
    """โครงข่าย LSTM แบบเต็ม"""

    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.cell = LSTMCell(input_size, hidden_size)
        self.W_out = np.random.randn(output_size, hidden_size) * 0.01
        self.b_out = np.zeros((output_size, 1))

    def forward(self, x_sequence):
        """การส่งผ่านสัญญาณไปข้างหน้าตลอดลำดับ"""
        h = np.zeros((self.hidden_size, 1))
        c = np.zeros((self.hidden_size, 1))

        self.gates_history = []

        for x in x_sequence:
            h, c, gates = self.cell.forward(x, h, c)
            self.gates_history.append(gates)

        # เอาต์พุตสุดท้าย
        y = self.W_out @ h + self.b_out
        return y, h

## 6. ทดสอบ LSTM

In [ ]:
# สร้าง LSTM
lstm = LSTM(input_size=2, hidden_size=4, output_size=1)

# ลำดับทดสอบ
sequence = [
    np.array([1, 0]),
    np.array([0, 1]),
    np.array([1, 1])
]

output, h_final = lstm.forward(sequence)

print("=== การส่งผ่านสัญญาณไปข้างหน้าของ LSTM ===")
print(f"\nสถานะแฝงสุดท้าย: {h_final.flatten()}")
print(f"เอาต์พุต: {output.flatten()}")

print("\n=== เกตในแต่ละขั้นเวลา ===")
for t, (f, i, o, c_tilde) in enumerate(lstm.gates_history):
    print(f"\nขั้นเวลาที่ {t+1}:")
    print(f"  เกตลืม: {f.flatten()}")
    print(f"  เกตอินพุต: {i.flatten()}")
    print(f"  เกตเอาต์พุต: {o.flatten()}")

## 7. การสร้างเซลล์ GRU (GRU Cell)

In [ ]:
class GRUCell:
    """
    เซลล์ GRU ที่มี 2 เกต:
    - เกตรีเซ็ต (Reset Gate): r_t = σ(W_r @ [h_{t-1}, x_t] + b_r)
    - เกตอัปเดต (Update Gate): z_t = σ(W_z @ [h_{t-1}, x_t] + b_z)

    การอัปเดตสถานะแฝง:
    - h̃_t = tanh(W_h @ [r_t * h_{t-1}, x_t] + b_h)
    - h_t = z_t * h_{t-1} + (1 - z_t) * h̃_t
    """

    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size

        # เกตรีเซ็ต
        self.W_r = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_r = np.zeros((hidden_size, 1))

        # เกตอัปเดต
        self.W_z = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_z = np.zeros((hidden_size, 1))

        # สถานะแฝงตัวเลือก
        self.W_h = np.random.randn(hidden_size, input_size + hidden_size) * 0.01
        self.b_h = np.zeros((hidden_size, 1))

    def forward(self, x, h_prev):
        """คำนวณหนึ่งขั้นเวลา"""
        combined = np.vstack([h_prev, x.reshape(-1, 1)])

        # เกตต่างๆ
        r_t = self._sigmoid(self.W_r @ combined + self.b_r)  # เกตรีเซ็ต
        z_t = self._sigmoid(self.W_z @ combined + self.b_z)  # เกตอัปเดต

        # สถานะแฝงตัวเลือก
        combined_reset = np.vstack([r_t * h_prev, x.reshape(-1, 1)])
        h_tilde = np.tanh(self.W_h @ combined_reset + self.b_h)

        # อัปเดตสถานะแฝง: z_t≈1 คงสถานะเดิม, z_t≈0 รับสถานะตัวเลือกเป็นหลัก (สัญนิยมตามบทที่ 7)
        h_t = z_t * h_prev + (1 - z_t) * h_tilde

        return h_t, (r_t, z_t, h_tilde)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


class GRU:
    """โครงข่าย GRU แบบเต็ม"""

    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)
        self.W_out = np.random.randn(output_size, hidden_size) * 0.01
        self.b_out = np.zeros((output_size, 1))

    def forward(self, x_sequence):
        h = np.zeros((self.hidden_size, 1))
        self.gates_history = []

        for x in x_sequence:
            h, gates = self.cell.forward(x, h)
            self.gates_history.append(gates)

        y = self.W_out @ h + self.b_out
        return y, h

## 8. เปรียบเทียบ RNN, LSTM และ GRU (เชิงคุณภาพ)

In [ ]:
# การเปรียบเทียบเชิงคุณภาพ: ทดสอบว่าแต่ละสถาปัตยกรรมรักษาข้อมูลจากขั้นต้นลำดับได้ดีเพียงใด
# งานสังเคราะห์: อินพุตขั้นแรกเป็นสัญญาณ +1 หรือ -1 ขั้นที่เหลือเป็น 0 ทั้งหมด
# เป้าหมายคือทำนายเครื่องหมายของสัญญาณขั้นแรกเมื่อถึงขั้นเวลาสุดท้าย
# หมายเหตุ: นี่เป็นการสาธิตเชิงคุณภาพด้วยการฝึกแบบเกรเดียนต์เชิงตัวเลข (หัวข้อ 2.3) ไม่ใช่การวัดประสิทธิภาพที่ครอบคลุมทุกงาน

def make_recall_dataset(n_sequences=8, seq_len=6, seed=0):
    signals = np.array(([1.0, -1.0] * ((n_sequences + 1) // 2))[:n_sequences])
    np.random.RandomState(seed).shuffle(signals)
    X = []
    for s in signals:
        seq = [np.array([s])] + [np.array([0.0])] * (seq_len - 1)
        X.append(seq)
    return X, signals

def predict_scalar(model, x_seq):
    result = model.forward(x_seq)
    y = result[0] if isinstance(result, tuple) else result[-1]
    return y[0, 0]

def mse_loss(model, X, y_true):
    preds = np.array([predict_scalar(model, seq) for seq in X])
    return np.mean((preds - y_true) ** 2)

def collect_arrays(obj):
    """เก็บ ndarray ทุกตัวของโมเดล (รวมถึงใน sub-object เช่น .cell) เพื่อใช้กับเกรเดียนต์เชิงตัวเลข"""
    arrays = []
    for name, val in vars(obj).items():
        if isinstance(val, np.ndarray):
            arrays.append(val)
        elif hasattr(val, "__dict__"):
            arrays.extend(collect_arrays(val))
    return arrays

def reinit_weights(model, scale, seed):
    """กำหนดค่าน้ำหนักเริ่มต้นใหม่ให้มีขนาดใหญ่พอที่สัญญาณอินพุตจะส่งผลต่อเอาต์พุตได้ตั้งแต่ต้น"""
    rng = np.random.RandomState(seed)
    for arr in collect_arrays(model):
        arr[:] = rng.randn(*arr.shape) * scale

def train_numerical_gradient(model, X, y_true, epochs=80, lr=0.3, eps=1e-3):
    """ฝึกด้วยเกรเดียนต์เชิงตัวเลข (numerical derivative) แทนการแพร่กระจายย้อนกลับ
    เพื่อให้ใช้ได้กับทั้ง RNN, LSTM และ GRU โดยไม่ต้องเขียน backward แยกแต่ละแบบ"""
    params = collect_arrays(model)
    loss_history = [mse_loss(model, X, y_true)]

    for epoch in range(epochs):
        for arr in params:
            flat = arr.reshape(-1)
            for idx in range(flat.size):
                original = flat[idx]

                flat[idx] = original + eps
                loss_plus = mse_loss(model, X, y_true)

                flat[idx] = original - eps
                loss_minus = mse_loss(model, X, y_true)

                flat[idx] = original
                grad = (loss_plus - loss_minus) / (2 * eps)
                flat[idx] = original - lr * grad

        loss_history.append(mse_loss(model, X, y_true))

    return loss_history

# เตรียมชุดข้อมูลเดียวกันสำหรับทั้ง 3 สถาปัตยกรรม
X_recall, y_recall = make_recall_dataset(n_sequences=8, seq_len=6, seed=0)

hidden_size = 3
rnn_cmp = SimpleRNN(input_size=1, hidden_size=hidden_size, output_size=1)
lstm_cmp = LSTM(input_size=1, hidden_size=hidden_size, output_size=1)
gru_cmp = GRU(input_size=1, hidden_size=hidden_size, output_size=1)

# กำหนดค่าน้ำหนักเริ่มต้นใหม่ให้มีขนาดใหญ่พอสำหรับสาธิต (ค่าเริ่มต้นปกติ *0.01 เล็กเกินกว่าจะเห็นผลใน 80 รอบ)
reinit_weights(rnn_cmp, scale=0.3, seed=1)
reinit_weights(lstm_cmp, scale=0.3, seed=1)
reinit_weights(gru_cmp, scale=0.3, seed=1)

loss_rnn = train_numerical_gradient(rnn_cmp, X_recall, y_recall)
loss_lstm = train_numerical_gradient(lstm_cmp, X_recall, y_recall)
loss_gru = train_numerical_gradient(gru_cmp, X_recall, y_recall)

plt.figure(figsize=(10, 6))
plt.plot(loss_rnn, label='RNN')
plt.plot(loss_lstm, label='LSTM')
plt.plot(loss_gru, label='GRU')
plt.xlabel('รอบการฝึก')
plt.ylabel('ค่าสูญเสีย (MSE)')
plt.title('เปรียบเทียบการลดลงของค่าสูญเสีย: งานจดจำสัญญาณขั้นต้นลำดับ (เชิงคุณภาพ)')
plt.legend()
plt.show()

print(f"ค่าสูญเสียสุดท้าย — RNN: {loss_rnn[-1]:.4f}, LSTM: {loss_lstm[-1]:.4f}, GRU: {loss_gru[-1]:.4f}")
print("หมายเหตุ: ผลนี้เป็นตัวอย่างเชิงคุณภาพจากงานสังเคราะห์ขนาดเล็กมากและการฝึกด้วยเกรเดียนต์เชิงตัวเลข ทั้งความเร็วและลำดับการลู่เข้าของแต่ละสถาปัตยกรรมขึ้นกับค่าเริ่มต้นและงานที่ทดสอบ จึงไม่ควรสรุปเป็นการเปรียบเทียบประสิทธิภาพทั่วไป")

## 9. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: สถานะแฝงของ RNN (RNN Hidden State)

In [ ]:
# ให้ RNN มีค่าดังนี้:
# W_xh = [[0.5]], W_hh = [[0.8]], b_h = [[0.1]]
# จงคำนวณสถานะแฝงสำหรับลำดับ x = [1, 2, 3]

W_xh = np.array([[0.5]])
W_hh = np.array([[0.8]])
b_h = np.array([[0.1]])

sequence = [1, 2, 3]
h = 0.0  # สถานะแฝงเริ่มต้น (สเกลาร์ เพราะทุกเมทริกซ์ในตัวอย่างนี้เป็น 1x1)

print("=== คำนวณสถานะแฝงของ RNN ===")
for t, x in enumerate(sequence):
    h = float(np.tanh(W_xh[0, 0] * x + W_hh[0, 0] * h + b_h[0, 0]))
    print(f"ขั้นเวลาที่ {t+1}: x={x}, h={h:.4f}")

### แบบฝึกหัดที่ 2: เกตของ LSTM (LSTM Gates)

In [ ]:
# ให้ผลลัพธ์เกตลืม = 0.8, ผลลัพธ์เกตอินพุต = 0.3
# สถานะเซลล์ก่อนหน้า = 1.0, สถานะเซลล์ตัวเลือก = 0.5
# จงคำนวณสถานะเซลล์ใหม่

f_t = 0.8  # เกตลืม
i_t = 0.3  # เกตอินพุต
c_prev = 1.0  # สถานะเซลล์ก่อนหน้า
c_tilde = 0.5  # สถานะเซลล์ตัวเลือก

c_new = f_t * c_prev + i_t * c_tilde
print(f"สถานะเซลล์ใหม่: {c_new:.4f}")
print(f"การคำนวณ: {f_t} × {c_prev} + {i_t} × {c_tilde} = {c_new:.4f}")

### แบบฝึกหัดที่ 3: การอัปเดตของ GRU (GRU Update)

In [ ]:
# ให้เกตอัปเดต = 0.7, สถานะแฝงก่อนหน้า = 0.5, สถานะแฝงตัวเลือก = 0.9
# จงคำนวณสถานะแฝงใหม่

z_t = 0.7  # เกตอัปเดต
h_prev = 0.5  # สถานะแฝงก่อนหน้า
h_tilde = 0.9  # สถานะแฝงตัวเลือก

h_new = z_t * h_prev + (1 - z_t) * h_tilde
print(f"สถานะแฝงใหม่: {h_new:.4f}")
print(f"การคำนวณ: {z_t} × {h_prev} + (1 - {z_t}) × {h_tilde} = {h_new:.4f}")

### แบบฝึกหัดที่ 4: คำนวณพารามิเตอร์

In [ ]:
# จงคำนวณจำนวนพารามิเตอร์ของ LSTM ที่มี input_size=100, hidden_size=128 (ตรงกับแบบฝึกหัดข้อ 2 ท้ายบท)

input_size = 100
hidden_size = 128

# LSTM มี 4 ชุดค่าน้ำหนัก (เกตลืม เกตอินพุต เกตเอาต์พุต และสถานะเซลล์ตัวเลือก)
# แต่ละชุด: W (hidden_size × (input_size + hidden_size)) + b (hidden_size)
params_per_gate = hidden_size * (input_size + hidden_size) + hidden_size
total_params = 4 * params_per_gate

print(f"พารามิเตอร์ของ LSTM:")
print(f"ขนาดอินพุต: {input_size}")
print(f"ขนาดชั้นซ่อน: {hidden_size}")
print(f"พารามิเตอร์ต่อเกต: {params_per_gate}")
print(f"พารามิเตอร์ LSTM ทั้งหมด: {total_params}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **RNN อย่างง่าย**: โครงสร้างพื้นฐาน มีปัญหาเกรเดียนต์สูญหายและเกรเดียนต์ระเบิด
2. **LSTM**: มี 3 เกต (ลืม อินพุต เอาต์พุต) ช่วยรักษาความสัมพันธ์ระยะยาว
3. **GRU**: มี 2 เกต (รีเซ็ต อัปเดต) พารามิเตอร์น้อยกว่า LSTM แต่ประสิทธิภาพใกล้เคียงกันในหลายงาน
4. **การเปรียบเทียบเชิงคุณภาพ**: สาธิตการฝึกทั้งสามสถาปัตยกรรมด้วยงานจดจำสัญญาณขั้นต้นลำดับ